# Time Series Analysis

---

### Table of Contents
1. Introduction and Setup
2. Time-Based Indexing and Selection
3. Resampling Time Series Data
4. Time Shifts and Rolling Windows

---

## 1. Introduction and Setup
- Pandas was originally developed for financial time series analysis and has
  incredibly powerful, built-in tools for working with dates and times.
- The key to unlocking these features is to use a `DatetimeIndex`.

In [15]:
import os
import pandas as pd

# --- Load the sample dataset with a DatetimeIndex ---
DATA_FOLDER = "pandas_data"
file_path = os.path.join(DATA_FOLDER, "sample_sales_data.csv")

try:
    # We instruct pandas to parse 'OrderDate' as dates and set it as the index.
    df = pd.read_csv(file_path, parse_dates=["OrderDate"], index_col="OrderDate")
    print("--- Sample Sales DataFrame with DatetimeIndex ---")
    print(df)
    print(f"\nType of the index: {type(df.index)}")  # Note the DatetimeIndex type

except FileNotFoundError:
    print(f"Error: The data file was not found at '{file_path}'")
    print("Please run '04_reading_and_writing_data.py' first to create it.")
    df = pd.DataFrame()  # Create an empty df to avoid further errors

--- Sample Sales DataFrame with DatetimeIndex ---
            OrderID   Product     Category   Price  Quantity
OrderDate                                                   
2025-01-15      101    Laptop  Electronics  1200.0         1
2025-01-15      102     Mouse  Electronics    25.5         2
2025-01-16      103  Keyboard  Electronics    75.0         1
2025-01-17      104   Monitor  Electronics   300.0         2
2025-01-18      105     Mouse  Accessories    27.0         3
2025-01-18      106    Webcam  Accessories    50.0         1

Type of the index: <class 'pandas.core.indexes.datetimes.DatetimeIndex'>



---

## 2. Time-Based Indexing and Selection
- With a DatetimeIndex, you can select and slice data in intuitive ways.

In [16]:
# --- Select all data for a specific day ---
print("Sales on 2025-01-15:\n", df.loc["2025-01-15"])

Sales on 2025-01-15:
             OrderID Product     Category   Price  Quantity
OrderDate                                                 
2025-01-15      101  Laptop  Electronics  1200.0         1
2025-01-15      102   Mouse  Electronics    25.5         2


In [17]:
# --- Select all data for a specific year or month ---
# Pandas is smart enough to interpret partial date strings.
print("\nAll sales in January 2025:\n", df.loc["2025-01"])


All sales in January 2025:
             OrderID   Product     Category   Price  Quantity
OrderDate                                                   
2025-01-15      101    Laptop  Electronics  1200.0         1
2025-01-15      102     Mouse  Electronics    25.5         2
2025-01-16      103  Keyboard  Electronics    75.0         1
2025-01-17      104   Monitor  Electronics   300.0         2
2025-01-18      105     Mouse  Accessories    27.0         3
2025-01-18      106    Webcam  Accessories    50.0         1


In [18]:
# --- Slicing a date range ---
start_date = "2025-01-16"
end_date = "2025-01-17"
print(f"\nSales between {start_date} and {end_date}:\n", df.loc[start_date:end_date])


Sales between 2025-01-16 and 2025-01-17:
             OrderID   Product     Category  Price  Quantity
OrderDate                                                  
2025-01-16      103  Keyboard  Electronics   75.0         1
2025-01-17      104   Monitor  Electronics  300.0         2



---

## 3. Resampling Time Series Data
- Resampling is the process of converting a time series from one frequency to another.
- Downsampling: Decreasing the frequency (e.g., daily to monthly). Requires aggregation.
- Upsampling: Increasing the frequency (e.g., daily to hourly). Requires filling/interpolation.
- The `.resample()` method works like .groupby() - you must chain an aggregation to it.
- Common Frequency Rules: 'D' (Day), 'W' (Week), 'M' (Month End), 'Q' (Quarter End), 'Y' (Year End)

In [19]:
# Our data is at a daily level. Let's downsample it to see total sales PER DAY.
# This will group all orders on the same day together.
daily_sales = df[["Price", "Quantity"]].resample("D").sum()

print("Total daily sales (resampled to 'D' frequency):\n", daily_sales)

Total daily sales (resampled to 'D' frequency):
              Price  Quantity
OrderDate                   
2025-01-15  1225.5         3
2025-01-16    75.0         1
2025-01-17   300.0         2
2025-01-18    77.0         4


*Note: Days with no sales (like Jan 16th in the original data) still appear in the resampled index, with an aggregated value of 0*.

---

## 4. Time Shifts and Rolling Windows
- These are common techniques for time series feature engineering.
- To better demonstrate, let's create a simple, continuous time series.

In [20]:
# A DataFrame with sales data for 7 consecutive days
date_range = pd.date_range(start="2025-01-01", periods=7, freq="D")
sales_data = pd.DataFrame({"Sales": [10, 15, 12, 18, 20, 25, 22]}, index=date_range)

print("New sample time series data:\n", sales_data)

New sample time series data:
             Sales
2025-01-01     10
2025-01-02     15
2025-01-03     12
2025-01-04     18
2025-01-05     20
2025-01-06     25
2025-01-07     22


In [21]:
# --- Shifting (`.shift()`) ---

# - Shifts data forward or backward by a specified number of periods.
# - Useful for calculating period-over-period changes.
sales_data["Previous_Day_Sales"] = sales_data["Sales"].shift(1)
print("\nData with previous day's sales (using .shift(1)):\n", sales_data)


Data with previous day's sales (using .shift(1)):
             Sales  Previous_Day_Sales
2025-01-01     10                 NaN
2025-01-02     15                10.0
2025-01-03     12                15.0
2025-01-04     18                12.0
2025-01-05     20                18.0
2025-01-06     25                20.0
2025-01-07     22                25.0


In [22]:
# --- Rolling Windows (`.rolling()`) ---

# - Calculates statistics over a "sliding window" of a fixed size.
# - Used for things like moving averages to smooth out data.
# - `.rolling(window=N)` creates a rolling view. You must chain an aggregation to it.
sales_data["3-Day_Moving_Average"] = sales_data["Sales"].rolling(window=3).mean()
print("\nData with 3-day moving average (using .rolling(3).mean()):\n", sales_data)


Data with 3-day moving average (using .rolling(3).mean()):
             Sales  Previous_Day_Sales  3-Day_Moving_Average
2025-01-01     10                 NaN                   NaN
2025-01-02     15                10.0                   NaN
2025-01-03     12                15.0             12.333333
2025-01-04     18                12.0             15.000000
2025-01-05     20                18.0             16.666667
2025-01-06     25                20.0             21.000000
2025-01-07     22                25.0             22.333333


*Note: The first two values are NaN because there are not enough preceding data points to fill the window of size 3.*

---

**Next:** [Basic Plotting](./12_basic_plotting.ipynb)